# Botometer testing

In [1]:
import os 
import sys
import json
import json
import botometer

In [2]:
sys.path.append("code/libs")
import ios

In [3]:
PATH_AUTH = 'auth'
TWITTER_APP_FN_AUTH = os.path.join(PATH_AUTH,'canpreg.json')

In [4]:
keys = ios.read_json(TWITTER_APP_FN_AUTH)
keys['rapidapi_key'] = ios.get_text(os.path.join(PATH_AUTH, 'rapidapi'), 'rapidapi_key_personal')

In [ ]:
# initialize Botometer class
rapidapi_key = keys['rapidapi_key']
twitter_app_auth = {
    'consumer_key': keys['consumer_key'],
    'consumer_secret': keys['consumer_secret'],
    'access_token': keys['access_token'],
    'access_token_secret': keys['access_token_secret']
}

bom = botometer.Botometer(wait_on_ratelimit=True,
                          rapidapi_key=rapidapi_key,
                          **twitter_app_auth)

# load the input
user_ids = ['BarackObama','elonmusk']

# start to check the bot scores
print(f"Check bot scores for {len(user_ids)} users")
bot_scores = []
for index, user_id in enumerate(user_ids):
    print(f"Checking user {user_id}, {index} / {len(user_ids)}...")
    try:
        result = bom.check_account(user_id)
        bot_scores.append(result)
    except KeyboardInterrupt:
        sys.exit()
    except Exception as e:
        err_msg = 'Error handling account {}\n{}: {}'.format(
            user_id,
            type(e).__name__,
            getattr(e, 'msg', '') or getattr(e, 'reason', ''),)
        print(err_msg)

In [25]:
print(f"Fetched bot scores for {len(bot_scores)} / {len(user_ids)} accounts")

Fetched bot scores for 2 / 2 accounts


___

# Botometer scores

In [1]:
import pandas as pd
import os
import sys
import glob
from pqdm.processes import pqdm

In [2]:
%reload_ext autoreload
%autoreload 2

sys.path.append("../code/libs")
import ios

In [31]:
ROOT = '../data'

## 1. Users to query

In [28]:
SBERT_SCORES_FN = os.path.join(ROOT, 'tweets_all_8clusters_mcs10_ss.pkl')
df_sbert = pd.read_pickle(SBERT_SCORES_FN)
df_sbert.head(2)

,id,author_id,corpus,score_q1,score_q2,score_q3,score_q4,score_q5,score_q6,score_q7,score_q8,total
0,1417673516414013440,2766734643,@loftmusik_ never been pregnant but if i were ...,0.732504,0.67069,0.760165,0.581427,0.681276,0.444975,0.526308,0.444595,4.841941
1,1417602329235361797,2422554078,@maryxwetzel i (my pregnant self) smoke weed i...,0.433795,0.577831,0.390623,0.492191,0.396159,0.437249,0.239965,0.279092,3.246906


In [29]:
user_lst = df_sbert.author_id.unique().astype(str)
ios.write_list_as_lines(user_lst, "../data/users_to_query_botometer_all.txt")
user_lst.shape
# 1384086

(1384086,)

In [33]:
thr = 0.42
query = ' or '.join([f"{c}>{thr}" for c in df_sbert.columns if c.startswith('score_q')])
print(query)
user_lst = df_sbert.query(query).author_id.unique().astype(str)
ios.write_list_as_lines(user_lst, f"../data/users_to_query_botometer_sbert{thr}.txt")
print(user_lst.shape)

score_q1>0.42 or score_q2>0.42 or score_q3>0.42 or score_q4>0.42 or score_q5>0.42 or score_q6>0.42 or score_q7>0.42 or score_q8>0.42
(367982,)


## 2. Botometer file (users with Botometer scores)

In [48]:
files = glob.glob("../data/botometer/*.json")
len(files)
# 217 076
# 217 877
# 220 505
# 220 689
# 220 949
# 223 375
# 229 322
# 350 675
# 350 674 (1 was private)

350674

In [49]:
def get_series(fn, english=True):
    obj = ios.read_json(fn, verbose=False)
    df = None
    if obj is not None:
        user = os.path.basename(fn).replace(".json","")
        val = 'english' if english else 'universal'
        df = pd.DataFrame({'user':user,'cap':obj['cap'][val],'raw':obj['raw_scores'][val]['overall'],'display':obj['display_scores'][val]['overall']}, index=[0])
    return df 

def get_empty_results(fn):
    obj = ios.read_json(fn, verbose=False)
    if obj is None:
        return fn
    return None

In [50]:
# test
get_series("../data/botometer/230882407.json")

,user,cap,raw,display
0,230882407,0.306188,0.04,0.2


In [ ]:
results = pqdm(files, get_series, n_jobs=20)
print(len(results))
_results = [r for r in results if r is not None]
print(len(_results))

# 350675 350384 350674

In [52]:
df_botometer = pd.concat(results)
df_botometer.reset_index(drop=True, inplace=True)
df_botometer.head()

,user,cap,raw,display
0,3020171467,0.792032,0.37,1.8
1,3787357817,0.419722,0.08,0.4
2,20563800,0.785221,0.33,1.6
3,2547515392,0.419722,0.08,0.4
4,153092097,0.802148,0.75,3.8


In [53]:
df_botometer.shape
# 229 322
# 350 384
# 350 674

(350674, 4)

In [54]:
# good results
ios.save_csv(df_botometer, "../data/users_botometer.csv")

In [ ]:
# users to query (errors in botometer)
empty_results = pqdm(files, get_empty_results, n_jobs=20)
empty_results = [r for r in empty_results if r is not None]
ios.write_list_as_lines([fn.split('/')[-1].replace('.json','') for fn in empty_results], "../data/users_to_query_botometer_empty.txt")
len(empty_results)
# 291

In [43]:
import os
for fn in empty_results:
    try:
        os.remove(fn)
    except:
        pass

In [46]:
!ls ../data/botometer | wc -lc

 350390 5649337


___

# All users

In [32]:
users_fn = os.path.join(ROOT, '../data/users_all.pkl')
df_users = pd.read_pickle(users_fn)
df_users.head()

,id,name,username,location,description,created_at,protected,verified
0,2766734643,cheyenne⛈,chyxendi,the USA unfortunately,20. capricorn. child advocate. nazi feminist. ...,2014-09-11 00:12:24,None,False
1,2422554078,jillian,spiritual_muma,"Ohio, USA",homebody 🍂,2014-04-01 19:16:20,None,False
2,166379492,Merry Stephanie 🎄,JetaimeLesMis,"Savannah, GA",RI➡️FL➡️GA. NE Patriots fan for LIFE; kicking ...,2010-07-14 00:49:06,None,False
3,64567001,Yazmin💋,Chef_Hernandezz,None,"Mom of 3 ❤️ TATTOOS 🖋 I'm a rebel, Idgaf. 🇲🇽#G...",2009-08-11 00:25:21,None,False
4,1337790916912947208,Voice of Africa Radio,atienoVoA,"Detroit, MI",A Public JournaL by ATIENO. an Expatriate. a T...,2020-12-12 16:06:36,None,False


In [33]:
empty_results[10]

'../data/botometer/319869296.json'

In [47]:
id=93106755
df_users.query("id==@id")

,id,name,username,location,description,created_at,protected,verified
2138607,93106755,Denny,DaddyDenDen,"Bronx, NY",Tweets about nothing. 🇬🇩,2009-11-28 03:03:37,None,False


In [22]:
df_users.shape

(3305111, 8)

In [27]:
!ls ../data/users | wc -lc

3305111 40203139


In [9]:
!ls ../data/botometer | wc -lc
# 217036 
# 218181

 218181 3524302


In [8]:
!ls ../cache/osm | wc -lc
# 57176
# 57184

  57184 2916343
